# Phase 4 Analysis Notebook

This notebook follows the two original Phase 4 study notes exactly in spirit and scope. It does **not** include later index-construction or custom block-collapse experiments.

The notebook reproduces:
- MD1: the rolling 52-week ownership-versus-sector correlation study
- MD2: the weekly cross-sectional variance-decomposition study

Data note:
- The analysis uses the repaired weekly panel built from the repo-local dataset plus historical repairs where needed.
- The repaired panel is used because the original panel had missing flagship names and contaminated group mappings.

## Workflow

1. Load the repaired weekly panel.
2. Run the MD1 rolling-correlation study and save its CSVs and charts.
3. Run the MD2 variance-decomposition study and save its CSVs and charts.
4. Pull the report-ready summaries back into the notebook for interpretation.

In [ ]:
from pathlib import Path
from IPython.display import Markdown, Image, display
import pandas as pd

from phase4.studies import Phase4Spec, run_phase4_package

In [ ]:
INPUT_PATH = Path('data/completed/weekly_panel_completed.csv')
OUTPUT_DIR = Path('phase4/results')

outputs = run_phase4_package(
    Phase4Spec(
        input_path=INPUT_PATH,
        output_dir=OUTPUT_DIR,
        rolling_window=52,
        rolling_min_obs=26,
        min_stocks=100,
        permutations=1000,
        seed=42,
        newey_west_lags=6,
        smooth_weeks=26,
    )
)

print('Phase 4 assets written to:', OUTPUT_DIR)

In [ ]:
panel = pd.read_csv(INPUT_PATH, parse_dates=['Date'])
panel_summary = pd.DataFrame(
    {
        'rows': [len(panel)],
        'dates': [panel['Date'].nunique()],
        'tickers': [panel['Ticker'].nunique()],
        'non_null_returns': [pd.to_numeric(panel['Weekly_Return'], errors='coerce').notna().sum()],
        'groups_present': [', '.join(sorted(panel['Promoter_Group'].dropna().unique()))],
    }
)
panel_summary

## MD1: Rolling Correlation Study

This section corresponds to the first original study note.

Core objects:
- baseline average pairwise correlation
- same-sector excess correlation
- same-group excess correlation
- group-by-group cohesion
- PSU-included and PSU-excluded variants
- a heuristic effective-bets diagnostic

In [ ]:
md1_summary = pd.read_csv(OUTPUT_DIR / 'md1_rolling_correlation' / 'summary.csv')
md1_summary

In [ ]:
md1_report = (OUTPUT_DIR / 'md1_rolling_correlation' / 'REPORT.md').read_text(encoding='utf-8')
display(Markdown(md1_report))

In [ ]:
display(Image(filename=str(OUTPUT_DIR / 'md1_rolling_correlation' / '01_core_comparison.png')))
display(Image(filename=str(OUTPUT_DIR / 'md1_rolling_correlation' / '02_psu_sensitivity.png')))
display(Image(filename=str(OUTPUT_DIR / 'md1_rolling_correlation' / '03_group_cohesion.png')))
display(Image(filename=str(OUTPUT_DIR / 'md1_rolling_correlation' / '04_effective_bets.png')))

## MD2: Weekly Variance Decomposition

This section corresponds to the second original study note.

Core objects:
- sector `R²` each week
- incremental group `R²` after sector removal
- coarse versus fine sector robustness
- with-PSU and ex-PSU robustness
- split-period and trend summaries
- group contribution decomposition

In [ ]:
md2_summary = pd.read_csv(OUTPUT_DIR / 'md2_variance_decomposition' / 'summary.csv')
md2_summary

In [ ]:
md2_report = (OUTPUT_DIR / 'md2_variance_decomposition' / 'REPORT.md').read_text(encoding='utf-8')
display(Markdown(md2_report))

In [ ]:
display(Image(filename=str(OUTPUT_DIR / 'md2_variance_decomposition' / '01_variance_decomposition.png')))
display(Image(filename=str(OUTPUT_DIR / 'md2_variance_decomposition' / '02_group_contributions.png')))

## Closing Read

The professional takeaway from Phase 4 is narrow and defensible:
- ownership and policy blocks create meaningful extra co-movement in the Indian market,
- but sector remains the larger organizing force,
- and the strong claim that the ownership factor has cleanly strengthened over time is not robust under the stricter fine-sector specification.

That is the correct stopping point for the two original notes.